In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [3]:
import numpy as np
import pandas as pd
import requests
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [4]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.5788221,
	"longitude": -122.4112033,
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation_probability", "precipitation", "rain", "is_day"],
	"timezone": "America/Los_Angeles",
	"past_days": 2,
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch",
	"forecast_hours": 12,
	"past_hours": 6,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(4).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(5).ValuesAsNumpy()
hourly_rain = hourly.Variables(6).ValuesAsNumpy()
hourly_is_day = hourly.Variables(7).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["is_day"] = hourly_is_day

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 47.555702209472656°N -122.38996887207031°E
Elevation: 0.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s

Hourly data
                         date  temperature_2m  relative_humidity_2m  \
0  2026-03-27 09:00:00+00:00       42.370701                  82.0   
1  2026-03-27 10:00:00+00:00       45.880699                  76.0   
2  2026-03-27 11:00:00+00:00       48.220699                  75.0   
3  2026-03-27 12:00:00+00:00       50.020699                  72.0   
4  2026-03-27 13:00:00+00:00       51.730701                  64.0   
5  2026-03-27 14:00:00+00:00       50.740700                  63.0   
6  2026-03-27 15:00:00+00:00       50.470699                  65.0   
7  2026-03-27 16:00:00+00:00       50.650700                  64.0   
8  2026-03-27 17:00:00+00:00       50.020699                  62.0   
9  2026-03-27 18:00:00+00:00       47.950699                  67.0   
10 2026-03-27 19:00:00+00:00       45.700699                

In [14]:
import sys
sys.path.insert(0, "..")
from src.weather_client import WeatherClient
from sqlalchemy import create_engine
from src.pipeline import create_tables, load_slow

In [15]:
client = WeatherClient()

In [10]:
!pip install jupysql

  Using cached jupysql-0.11.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached jupysql_plugin-0.4.5-py3-none-any.whl.metadata (7.8 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
Using cached jupysql-0.11.1-py3-none-any.whl (95 kB)
Using cached jupysql_plugin-0.4.5-py3-none-any.whl (192 kB)
Using cached backoff-2.2.1-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.8/665.8 kB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [jupysql]7/10 [ploomber-core]


In [17]:
%load_ext sql

In [21]:
engine = create_engine("sqlite:///test.db")

In [22]:
create_tables(engine)

In [ ]:
load_slow(